# OASST BM25 Router Playground

A small notebook for inspecting individual OASST scenarios against the current local-anchor criteria cards and the raw-criteria baseline.

Use this to answer: *what does the open BM25 gate retrieve for this scenario, and where do cards vs raw criteria disagree?*


## Setup

Run these first. The notebook auto-finds the repo root whether the kernel starts in the repo root or in `notebooks/`.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    def Markdown(text: str) -> str:
        return text

    def display(obj) -> None:
        print(obj)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scenario_classifier" / "audits" / "bm25_audit.py").exists():
            return candidate
    raise RuntimeError(f"Could not find EigenBench repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scenario_classifier.retrieval.retrievers import BM25Retriever as BM25Index  # noqa: E402
from scenario_classifier.core.io import (  # noqa: E402
    DEFAULT_CARD_CORPUS,
    DEFAULT_RAW_CORPUS,
    DEFAULT_SCENARIOS,
    load_jsonl,
    load_scenarios,
)
from scenario_classifier.audits.bm25_audit import (  # noqa: E402
    flag,
    margin,
    positive_candidates,
    summarize,
)

TOP_K = 8

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.width", 140)

REPO_ROOT


## Load Data

This loads OASST scenarios, current v0.4 cards, current raw criteria, and the precomputed audit rows if present.


In [ ]:
SCENARIOS_PATH = DEFAULT_SCENARIOS
CARD_CORPUS_PATH = DEFAULT_CARD_CORPUS
RAW_CORPUS_PATH = DEFAULT_RAW_CORPUS
AUDIT_PATH = REPO_ROOT / "data" / "output" / "router_audit" / "oasst_bm25_local_anchors.jsonl"
SUMMARY_PATH = REPO_ROOT / "data" / "output" / "router_audit" / "oasst_bm25_local_anchors_summary.json"

oasst = load_scenarios(SCENARIOS_PATH)
cards = load_jsonl(CARD_CORPUS_PATH)
raw_criteria = load_jsonl(RAW_CORPUS_PATH)
audit_rows = load_jsonl(AUDIT_PATH) if AUDIT_PATH.exists() else []
audit_summary = json.loads(SUMMARY_PATH.read_text()) if SUMMARY_PATH.exists() else None

card_index = BM25Index(cards)
raw_index = BM25Index(raw_criteria)

summary_df = pd.DataFrame([
    {
        "oasst_scenarios": len(oasst),
        "card_docs": len(cards),
        "raw_docs": len(raw_criteria),
        "audit_rows": len(audit_rows),
        "default_top_k": TOP_K,
    }
])
display(summary_df)

if audit_summary:
    display(pd.DataFrame([audit_summary]))


## Inspect The Corpora

Quick checks for what is inside the card corpus and raw-criteria baseline.


In [ ]:
def corpus_frame(docs: list[dict], *, raw: bool = False) -> pd.DataFrame:
    text_field = "criterion_text" if raw else "claim"
    rows = []
    for doc in docs:
        rows.append({
            "constitution": doc.get("constitution", ""),
            "criterion_id": doc.get("criterion_id", ""),
            "text": doc.get(text_field) or doc.get("embedding_text", ""),
        })
    return pd.DataFrame(rows)

card_df = corpus_frame(cards)
raw_df = corpus_frame(raw_criteria, raw=True)

display(card_df.groupby("constitution").size().rename("cards").reset_index())
display(raw_df.groupby("constitution").size().rename("raw_criteria").reset_index())


In [ ]:
# Browse cards or raw criteria by constitution.
constitution = "deep_ecology"

display(card_df[card_df["constitution"] == constitution].reset_index(drop=True))
display(raw_df[raw_df["constitution"] == constitution].reset_index(drop=True))


## Router Helpers

`route_oasst(index)` inspects one OASST scenario. `route_text(text)` lets you type any scenario manually.


In [ ]:
def rank_frame(items: list[dict]) -> pd.DataFrame:
    rows = []
    for item in items:
        rows.append({
            "rank": item.get("rank"),
            "score": item.get("score"),
            "constitution": item.get("constitution", ""),
            "criterion_id": item.get("criterion_id", ""),
            "matched_terms": ", ".join(item.get("matched_terms", [])),
            "text": item.get("text", ""),
        })
    return pd.DataFrame(rows, columns=["rank", "score", "constitution", "criterion_id", "matched_terms", "text"])


def route_text(text: str, *, top_k: int = TOP_K, show: bool = True) -> dict:
    card_top = card_index.top_k(text, top_k)
    raw_top = raw_index.top_k(text, top_k)
    candidates = positive_candidates(card_top, raw_top)
    row = {
        "scenario": text,
        "flag": flag(card_top, raw_top),
        "gate_decision": "pass_bm25" if candidates else "embedding_fallback",
        "candidate_count": len(candidates),
        "candidate_constitutions": sorted({str(item["constitution"]) for item in candidates}),
        "candidate_criteria": [item["criterion_id"] for item in candidates],
        "card_top_constitution": card_top[0]["constitution"] if card_top else "",
        "card_top_score": card_top[0]["score"] if card_top else 0.0,
        "card_margin": margin(card_top),
        "raw_top_constitution": raw_top[0]["constitution"] if raw_top else "",
        "raw_top_score": raw_top[0]["score"] if raw_top else 0.0,
        "raw_margin": margin(raw_top),
        "card_top": card_top,
        "raw_top": raw_top,
    }
    if show:
        display(Markdown(f"### Scenario\n> {text}"))
        display(pd.DataFrame([{k: v for k, v in row.items() if k not in {"scenario", "card_top", "raw_top"}}]))
        display(Markdown("#### Card corpus"))
        display(rank_frame(card_top))
        display(Markdown("#### Raw criteria baseline"))
        display(rank_frame(raw_top))
    return row


def route_oasst(index: int, *, top_k: int = TOP_K, show: bool = True) -> dict:
    return route_text(oasst[int(index)], top_k=top_k, show=show)


def search_oasst(query: str, *, limit: int = 25) -> pd.DataFrame:
    q = query.lower()
    rows = [
        {"scenario_index": i, "scenario": scenario}
        for i, scenario in enumerate(oasst)
        if q in scenario.lower()
    ]
    return pd.DataFrame(rows[:limit])


def audit_slice(
    *,
    flag_name = None,
    gate = None,
    contains = None,
    limit: int = 25,
) -> pd.DataFrame:
    rows = audit_rows
    if flag_name:
        rows = [row for row in rows if row.get("flag") == flag_name]
    if gate:
        rows = [row for row in rows if row.get("gate_decision") == gate]
    if contains:
        q = contains.lower()
        rows = [row for row in rows if q in row.get("scenario", "").lower()]
    fields = [
        "scenario_index",
        "flag",
        "gate_decision",
        "candidate_count",
        "candidate_constitutions",
        "card_top_constitution",
        "card_top_score",
        "raw_top_constitution",
        "raw_top_score",
        "scenario",
    ]
    return pd.DataFrame([{field: row.get(field) for field in fields} for row in rows[:limit]])


## Try Individual OASST Scenarios

Change the index or search first, then route the scenario.


In [ ]:
search_oasst("climate", limit=15)


In [ ]:
route_oasst(84)


## Try Your Own Scenario

Type a scenario-like user question and inspect what the open BM25 gate lets through.


In [ ]:
custom_scenario = "Should I have fewer children because the planet is already crowded?"

route_text(custom_scenario)


## Look At Audit Buckets

Useful buckets: `both_agree`, `card_only`, `raw_only`, `disagree`, `no_match`. `no_match` means BM25 found no positive lexical overlap, so the row should go to embedding fallback.


In [ ]:
audit_slice(flag_name="disagree", limit=20)


In [ ]:
audit_slice(gate="embedding_fallback", contains="climate", limit=20)


## Notes

- BM25 here is only an open lexical gate. Treat retrieved items as candidates, not final labels.
- False positives are acceptable at this stage; false negatives are the thing to watch.
- `card_only`, `raw_only`, and `disagree` rows are the most useful places to debug whether the card compression lost or improved recall.
